In [ ]:
# 1. Install dependencies
!pip install -q pycryptodome msgpack lz4

import os
from google.colab import files

# 2. Download only the required 3 files
!wget -q https://raw.githubusercontent.com/NavHobbyDev/lunar-tear-masterdata-patcher/main/patch_masterdata.py
!wget -q https://raw.githubusercontent.com/NavHobbyDev/lunar-tear-masterdata-patcher/main/run_patches.py
!wget -q https://raw.githubusercontent.com/NavHobbyDev/lunar-tear-masterdata-patcher/main/config.json

# 3. Upload the source file (must be named origin.bin.e)
print("Upload origin.bin.e:")
uploaded = files.upload()

# Rename if uploaded under a different name
for name in uploaded:
    if name != "origin.bin.e":
        os.rename(name, "origin.bin.e")
        print(f"Renamed: {name} → origin.bin.e")

# 4. Store file/directory state BEFORE running the patcher
before_files = set(os.listdir('.'))
before_advance = set(os.listdir('advance')) if os.path.exists('advance') else set()

# 5. Run the main patching script
!python run_patches.py

# 6. Detect newly created files/directories AFTER execution
after_files = set(os.listdir('.'))
new_items = after_files - before_files

after_advance = set(os.listdir('advance')) if os.path.exists('advance') else set()
new_advance = after_advance - before_advance

# 7. Automatically download results based on detected changes
if os.path.exists('advance') and (new_advance or 'advance' in new_items):
    print("\n[Auto-download] Mode 3 (advance) detected. Zipping and downloading...")
    !zip -q -r advance.zip advance/
    files.download('advance.zip')

elif len([f for f in new_items if f.endswith('.bin.e')]) > 1:
    print("\n[Auto-download] Multiple .bin.e files detected (Mode 2). Zipping...")
    !zip -q -r results.zip *.bin.e
    files.download('results.zip')

elif any(f.endswith('.bin.e') for f in new_items):
    new_bin_files = [f for f in new_items if f.endswith('.bin.e')]
    for bin_file in new_bin_files:
        print(f"\n[Auto-download] Downloading file: {bin_file}")
        files.download(bin_file)

else:
    print("\n[Auto-download] No new .bin.e files or advance/ directory found.")